# 🌧️ MLOps — Rain in Australia Prediction
**Project:** [data-guru0/MLOPS-with-KubeHunter-and-KubeBench](https://github.com/data-guru0/MLOPS-with-KubeHunter-and-KubeBench)

### Pipeline Overview
1. **Environment Setup** — secrets, packages, repo clone
2. **Data Ingestion** — pull raw data from MongoDB Atlas
3. **Data Validation** — schema + null checks
4. **Data Transformation** — encoding, scaling, train/test split
5. **Model Training** — Random Forest / GradientBoosting
6. **Model Evaluation** — accuracy, F1, ROC-AUC
7. **MLflow Tracking** — log params, metrics, model artifact to DagsHub
8. **Artifact Export** — save model.pkl to `artifacts/models/`

## 0️⃣  Environment Variables & Secrets

In [ ]:
import os
from google.colab import userdata

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ updated

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

## 1️⃣  Install Dependencies

In [ ]:
%%capture
!pip install pymongo dnspython mlflow dagshub scikit-learn pandas numpy joblib imbalanced-learn

## 2️⃣  Clone Repository & Set Working Directory

In [ ]:
import os

REPO_URL  = "https://github.com/data-guru0/MLOPS-with-KubeHunter-and-KubeBench.git"
REPO_NAME = "MLOPS-with-KubeHunter-and-KubeBench"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C {REPO_NAME} pull

os.chdir(REPO_NAME)
print("📁 Working directory:", os.getcwd())
!ls

## 3️⃣  Create Required Directories

In [ ]:
import os

dirs = [
    "artifacts/raw",
    "artifacts/processed",
    "artifacts/models",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  ✅ {d}")

## 4️⃣  Data Ingestion — Pull from MongoDB Atlas

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient

MONGO_URL = os.environ['MONGO_DB_URL']
DB_NAME   = "MLOPS"           # adjust if your DB is named differently
COL_NAME  = "rain-dataset"    # adjust to your collection name

print(f"🔌 Connecting to MongoDB …")
client = MongoClient(MONGO_URL)
db     = client[DB_NAME]
col    = db[COL_NAME]

print(f"📥 Fetching documents from '{DB_NAME}.{COL_NAME}' …")
docs = list(col.find({}, {"_id": 0}))
df   = pd.DataFrame(docs)

print(f"✅ Loaded {len(df):,} rows × {len(df.columns)} cols")
df.to_csv("artifacts/raw/rain_data.csv", index=False)
print("   Saved → artifacts/raw/rain_data.csv")
df.head()

### 4a. Fallback — Download Kaggle dataset if MongoDB is empty

In [ ]:
# Run this cell ONLY if the MongoDB collection is empty or not yet populated.
# It downloads the Rain-in-Australia dataset directly from a public URL.

if len(df) == 0:
    print("⚠️  MongoDB collection empty — downloading public dataset …")
    url = (
        "https://raw.githubusercontent.com/dsrscientist/dataset1/master/"
        "weatherAUS.csv"
    )
    df = pd.read_csv(url)
    df.to_csv("artifacts/raw/rain_data.csv", index=False)
    print(f"✅ Downloaded {len(df):,} rows")
else:
    print("✅ Using MongoDB data — skipping fallback download")

## 5️⃣  Data Validation

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("artifacts/raw/rain_data.csv")

EXPECTED_COLS = [
    'Location','MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine',
    'WindGustDir','WindGustSpeed','WindDir9am','WindDir3pm',
    'WindSpeed9am','WindSpeed3pm','Humidity9am','Humidity3pm',
    'Pressure9am','Pressure3pm','Cloud9am','Cloud3pm','Temp9am','Temp3pm',
    'RainToday','RainTomorrow'
]

# ── 1. Column presence check ─────────────────────────────────────────────────
missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
if missing_cols:
    print(f"⚠️  Missing columns: {missing_cols}")
else:
    print("✅ All expected columns present")

# ── 2. Null summary ──────────────────────────────────────────────────────────
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("\n📊 Null % per column (top 10):")
print(null_pct[null_pct > 0].head(10).to_string())

# ── 3. Target distribution ───────────────────────────────────────────────────
if 'RainTomorrow' in df.columns:
    dist = df['RainTomorrow'].value_counts(normalize=True) * 100
    print(f"\n🎯 Target distribution:\n{dist.to_string()}")

print(f"\nShape: {df.shape}")

## 6️⃣  Data Transformation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

df = pd.read_csv("artifacts/raw/rain_data.csv")

# ── 1. Drop rows where target is null ────────────────────────────────────────
df.dropna(subset=['RainTomorrow'], inplace=True)

# ── 2. Parse Date → Year / Month / Day ───────────────────────────────────────
if 'Date' in df.columns:
    df['Date']  = pd.to_datetime(df['Date'], errors='coerce')
    df['Year']  = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day']   = df['Date'].dt.day
    df.drop(columns=['Date'], inplace=True)

# ── 3. Encode binary yes/no columns ──────────────────────────────────────────
for col in ['RainToday', 'RainTomorrow']:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0})

# ── 4. Encode categorical columns ────────────────────────────────────────────
cat_cols = ['Location','WindGustDir','WindDir9am','WindDir3pm']
le_dict  = {}
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

# ── 5. Fill remaining NaNs with median ───────────────────────────────────────
df.fillna(df.median(numeric_only=True), inplace=True)

# ── 6. Feature / Target split ────────────────────────────────────────────────
FEATURES = [
    'Location','MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine',
    'WindGustDir','WindGustSpeed','WindDir9am','WindDir3pm',
    'WindSpeed9am','WindSpeed3pm','Humidity9am','Humidity3pm',
    'Pressure9am','Pressure3pm','Cloud9am','Cloud3pm','Temp9am',
    'Temp3pm','RainToday','Year','Month','Day'
]
# Keep only features that exist
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'RainTomorrow'

X = df[FEATURES]
y = df[TARGET]

# ── 7. Scale numerical features ──────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES)

# ── 8. Train / Test split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# ── 9. Persist artifacts ─────────────────────────────────────────────────────
X_train.to_csv('artifacts/processed/X_train.csv', index=False)
X_test.to_csv('artifacts/processed/X_test.csv',  index=False)
y_train.to_csv('artifacts/processed/y_train.csv', index=False)
y_test.to_csv('artifacts/processed/y_test.csv',  index=False)
joblib.dump(scaler,  'artifacts/processed/scaler.pkl')
joblib.dump(le_dict, 'artifacts/processed/label_encoders.pkl')

print(f"✅ Transformation complete")
print(f"   Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"   Features used: {FEATURES}")

## 7️⃣  Model Training + MLflow Logging

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)

# ── Load processed data ───────────────────────────────────────────────────────
X_train = pd.read_csv('artifacts/processed/X_train.csv')
X_test  = pd.read_csv('artifacts/processed/X_test.csv')
y_train = pd.read_csv('artifacts/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('artifacts/processed/y_test.csv').squeeze()

# ── MLflow tracking URI ───────────────────────────────────────────────────────
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("Rain-Australia-MLOps")

# ── Model configs to try ──────────────────────────────────────────────────────
MODEL_CONFIGS = [
    {
        "name": "RandomForest",
        "model": RandomForestClassifier(
            n_estimators=100, max_depth=10,
            min_samples_split=5, random_state=42, n_jobs=-1
        ),
        "params": {"n_estimators": 100, "max_depth": 10, "min_samples_split": 5}
    },
    {
        "name": "GradientBoosting",
        "model": GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1,
            max_depth=5, random_state=42
        ),
        "params": {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 5}
    },
]

best_model      = None
best_f1         = -1
best_model_name = ""

for cfg in MODEL_CONFIGS:
    with mlflow.start_run(run_name=cfg["name"]):
        print(f"\n🏋️  Training {cfg['name']} …")

        # Log hyperparameters
        mlflow.log_params(cfg["params"])
        mlflow.log_param("model_type", cfg["name"])

        # Train
        clf = cfg["model"]
        clf.fit(X_train, y_train)

        # Evaluate
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        acc   = accuracy_score(y_test, y_pred)
        f1    = f1_score(y_test, y_pred)
        prec  = precision_score(y_test, y_pred)
        rec   = recall_score(y_test, y_pred)
        auc   = roc_auc_score(y_test, y_prob)

        # Log metrics
        mlflow.log_metrics({
            "accuracy":  acc,
            "f1_score":  f1,
            "precision": prec,
            "recall":    rec,
            "roc_auc":   auc,
        })

        # Log model artifact
        mlflow.sklearn.log_model(
            clf,
            artifact_path="model",
            registered_model_name=f"RainAustralia-{cfg['name']}"
        )

        print(f"   Accuracy : {acc:.4f}")
        print(f"   F1 Score : {f1:.4f}")
        print(f"   Precision: {prec:.4f}")
        print(f"   Recall   : {rec:.4f}")
        print(f"   ROC-AUC  : {auc:.4f}")

        # Track best model
        if f1 > best_f1:
            best_f1         = f1
            best_model      = clf
            best_model_name = cfg["name"]

print(f"\n🏆 Best model: {best_model_name}  (F1={best_f1:.4f})")

## 8️⃣  Model Evaluation — Detailed Report

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Classification Report — {best_model_name}")
print(classification_report(y_test, y_pred, target_names=["No Rain", "Rain"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No Rain", "Rain"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix — {best_model_name}")

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name=best_model_name)
axes[1].set_title("ROC Curve")
axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].legend()

plt.tight_layout()
plt.savefig("artifacts/models/evaluation_plots.png", dpi=150)
plt.show()
print("📊 Plots saved → artifacts/models/evaluation_plots.png")

## 9️⃣  Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(
        best_model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    feat_imp.head(15).plot(kind='barh', color='steelblue')
    plt.title(f"Top-15 Feature Importances — {best_model_name}")
    plt.xlabel("Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig("artifacts/models/feature_importance.png", dpi=150)
    plt.show()
    print("📊 Feature importance plot saved")
else:
    print("ℹ️  Model does not expose feature_importances_")

## 🔟  Save Best Model as `model.pkl`

In [ ]:
import joblib

MODEL_PATH = "artifacts/models/model.pkl"
joblib.dump(best_model, MODEL_PATH)
print(f"✅ Best model ({best_model_name}) saved → {MODEL_PATH}")

# Quick smoke-test
loaded = joblib.load(MODEL_PATH)
sample = X_test.iloc[:3]
preds  = loaded.predict(sample)
labels = {0: "No Rain", 1: "Rain"}
print("\n🔍 Smoke-test predictions on 3 samples:")
for i, p in enumerate(preds):
    print(f"   Sample {i+1}: {labels[p]}")

## 1️⃣1️⃣  (Optional) Log Evaluation Plots to MLflow

In [ ]:
import mlflow
import os

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("Rain-Australia-MLOps")

with mlflow.start_run(run_name=f"BestModel-{best_model_name}-Artifacts"):
    mlflow.log_artifact("artifacts/models/evaluation_plots.png")
    if os.path.exists("artifacts/models/feature_importance.png"):
        mlflow.log_artifact("artifacts/models/feature_importance.png")
    mlflow.log_artifact(MODEL_PATH)
    mlflow.log_metric("best_f1", best_f1)
    mlflow.log_param("best_model", best_model_name)

print("✅ Artifacts logged to MLflow / DagsHub")

## 1️⃣2️⃣  (Optional) Push Raw Data to MongoDB
Use this if you need to seed your MongoDB Atlas collection for the first time.

In [ ]:
# ⚠️  Run ONLY if your MongoDB collection is empty and you want to seed it.
SEED_MONGODB = False   # ← change to True to seed

if SEED_MONGODB:
    import pandas as pd
    from pymongo import MongoClient
    import os

    # Download public dataset
    url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/weatherAUS.csv"
    seed_df = pd.read_csv(url)
    print(f"📥 Downloaded {len(seed_df):,} rows")

    # Push to MongoDB
    client = MongoClient(os.environ['MONGO_DB_URL'])
    db     = client["MLOPS"]
    col    = db["rain-dataset"]

    records = seed_df.where(pd.notnull(seed_df), None).to_dict('records')
    col.insert_many(records)
    print(f"✅ Inserted {len(records):,} documents into MongoDB")
else:
    print("Skipping MongoDB seed (SEED_MONGODB=False)")

## 1️⃣3️⃣  Summary

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib, os

model  = joblib.load("artifacts/models/model.pkl")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("═" * 50)
print("  🌧️  Rain in Australia — MLOps Pipeline Summary")
print("═" * 50)
print(f"  Best Model    : {best_model_name}")
print(f"  Accuracy      : {accuracy_score(y_test, y_pred):.4f}")
print(f"  F1 Score      : {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC       : {roc_auc_score(y_test, y_prob):.4f}")
print(f"  Model saved   : artifacts/models/model.pkl")
print(f"  MLflow URI    : {os.environ['MLFLOW_TRACKING_URI']}")
print("═" * 50)